# BUFS 다국어 입학 도우미 — LangChain & RAG

- **작성자:** 맥슈웰 데이브
- **학번:** 20232829
- **프로젝트 유형:** 다국어 특화 실용 AI 서비스
- **핵심 기술:** LangChain, RAG, Chroma VectorDB, OpenAI Embedding/LLM, Streamlit

한국어·영어로 작성된 부산외국어대학교 공식 입학 문서를 검색 증강 생성(RAG)으로 연결하여, 사용자가 원하는 언어로 근거와 출처가 포함된 답변을 제공하고 개인화된 제출서류 체크리스트를 생성합니다.

## 1. 서비스 구조

1. PDF 문서를 페이지별로 추출하고 약 2,000자 단위로 분할합니다.
2. 각 청크에 파일명, 페이지, 문서 언어, 지원 과정 메타데이터를 부여합니다.
3. `text-embedding-3-small`로 임베딩하여 Chroma VectorDB에 저장합니다.
4. 다국어 질문과 의미적으로 가까운 문서를 검색합니다.
5. `gpt-4o-mini`가 검색된 공식 문서만 사용하여 질문 언어로 답변합니다.
6. 일반 Q&A 외에도 지원 과정별 맞춤 제출서류 체크리스트를 생성합니다.

In [ ]:
# 작성자: 맥슈웰 데이브 / 학번: 20232829
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "submission":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import CHAT_MODEL, EMBED_MODEL, CHROMA_DIR
print({"chat_model": CHAT_MODEL, "embedding_model": EMBED_MODEL, "chroma": str(CHROMA_DIR)})

## 2. 공식 PDF 추출 및 메타데이터 확인

In [ ]:
from collections import Counter
from extract import extract_all

chunks = extract_all()
print(f"전체 청크 수: {len(chunks)}")
for source, count in Counter(chunk.source for chunk in chunks).items():
    print(f"{count:>2} | {source}")

## 3. VectorDB 구축

아래 셀은 PDF가 변경되었을 때만 실행합니다. 안정적인 해시 ID를 사용하므로 재실행해도 벡터가 중복되지 않습니다.

In [ ]:
# OPENAI_API_KEY가 설정되어 있을 때 실행
# from ingest import build
# print(f"인덱싱 완료: {build()}개 청크")

## 4. 다국어 검색

동일한 한국어·영어 원문 데이터베이스를 대상으로 영어, 중국어, 베트남어 등 다양한 언어의 질문을 검색할 수 있습니다.

In [ ]:
from retriever import retrieve

question = "本科申请需要什么材料?"
documents = retrieve(question, k=4, level="undergraduate")
for document in documents:
    print(document.metadata["source"], "p.", document.metadata["page"])

## 5. 근거 기반 다국어 답변

검색 결과가 없으면 LLM을 호출하지 않고 입학처 확인을 안내합니다. 검색 결과가 있으면 공식 문맥만 사용하고 파일명과 페이지를 인용합니다.

In [ ]:
from answerer import answer

result = answer(question, documents)
print(result["text"])
print("\nCitations:", result["citations"])

## 6. 개인화 제출서류 체크리스트

단순 번역을 넘어 지원 과정, 전공, 신입·편입 여부, 답변 언어를 반영한 실행 가능한 체크리스트를 생성합니다.

In [ ]:
from checklist import build_checklist

checklist_result = build_checklist(
    level="undergraduate",
    program="Korean Language",
    applicant_type="new",
    lang="English",
)
print(checklist_result["text"])
print("\nCitations:", checklist_result["citations"])

## 7. 검증 및 웹 실행

```powershell
py -3 -m pytest
py -3 eval/run_eval.py
py -3 -m streamlit run app.py
```

개발 완료 시점 기준 단위 테스트 12개 및 한국어·영어·중국어·베트남어 라이브 평가 4개를 모두 통과했습니다.